# Let's try some machine learning!

## Create the Neural Network

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import src.machine_learning as ML

class GlyphClassifier(nn.Module):
    def __init__(self, NUM_classes, resolution=(64, 64)):
        super(GlyphClassifier, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * resolution[0]//4 * resolution[1]//4, 128)
        self.fc2 = nn.Linear(128, NUM_classes)  

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # [B, 32, 32, 32]
        x = self.pool(F.relu(self.conv2(x)))  # [B, 64, 16, 16]
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x  

In [ ]:
NUM_classes = 20 #Number of classes can be set here

# Getting the dataset
dataset_file = 'data/simple-star.zip'
train_dataset = ML.GlyphDataset(dataset_file, resize=(16, 16), split = "train", num_classes=NUM_classes)
test_dataset = ML.GlyphDataset(dataset_file, resize=(16, 16), split = 'test', num_classes=NUM_classes)
# train_dataset = ML.GlyphDataset('data/simple-star-L.zip', split = "train", num_classes=NUM_classes)
# test_dataset = ML.GlyphDataset('data/simple-star-L.zip', split = 'test', num_classes=NUM_classes)

# Assign the loaders 

train_loader = ML.create_loader(train_dataset, batch_size=64, shuffle = True)
test_loader = ML.create_loader(test_dataset, batch_size=64, shuffle = False)

In [ ]:
import torch
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = GlyphClassifier(NUM_classes, resolution=(16, 16)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 100
losses = []

for epoch in range(num_epochs):
    model.train()
    for images, labels, _ in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {losses[-1]:.4f}")


In [ ]:
ML.plot_training_loss(losses)

In [ ]:
all_true_labels = []
all_predicted_labels = []

model.eval()
correct_predictions = 0
total_samples = 0
with torch.no_grad():
    for images, labels, _ in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted_labels = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted_labels == labels).sum().item()
        all_true_labels.extend(labels.cpu().numpy())
        all_predicted_labels.extend(predicted_labels.cpu().numpy())



accuracy = 100 * correct_predictions / total_samples
print(f'\n--- Evaluation Results ---')
print(f'Total Test Samples: {total_samples}')
print(f'Correct Predictions: {correct_predictions}')
print(f'Accuracy on the test set: {accuracy:.2f}%')
print(f'--------------------------')



class_names = [f'Class {i}' for i in range(NUM_classes)]

print("\nGenerating Confusion Matrix...")
ML.plot_confusion_matrix(
    y_true=all_true_labels,
    y_pred=all_predicted_labels,
    classes=class_names,
    normalize=False
)
plt.show() 



In [ ]:
ML.show_incorrect_predictions(model, test_loader, num_classes=NUM_classes, max_display=10, device=device)

## **More Complicated Training** 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import src.machine_learning as ML

class GlyphClassifier(nn.Module):
    def __init__(self, NUM_classes, resolution):
        super(GlyphClassifier, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * resolution[0]//4 * resolution[1]//4, 128),
            nn.ReLU(),
            nn.Linear(128, NUM_classes)
        )
 
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten the output
        x = self.classifier(x)
        return x

In [ ]:
def train_model(model, train_loader, num_epochs=100, lr=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for images, labels, _ in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")
    return model, losses

In [ ]:
import pandas as pd


results = []

dataset_file = 'data/random-stars-L.zip'

for nclasses in [10, 20, 50, 80, 100, 150]:
    for res in [32, 48, 64, 128, 198, 256]:
        print(f"\n=== Training for {nclasses} classes at {res}x{res} resolution ===")

        train_dataset = ML.GlyphDataset(dataset_file, split='train', resize=(res, res), num_classes=nclasses)
        test_dataset = ML.GlyphDataset(dataset_file, split='test', resize=(res, res), num_classes=nclasses)

        train_loader = ML.create_loader(train_dataset, batch_size=64, shuffle=True)
        test_loader = ML.create_loader(test_dataset, batch_size=64, shuffle=False)

        model = GlyphClassifier(nclasses, resolution=(res, res))
        trained_model, loss_history = train_model(model, train_loader, num_epochs=100, lr=0.001)

        # Save model
        model_path = f'model_n{nclasses}_res{res}.pth'
        torch.save(trained_model.state_dict(), model_path)

        # === Evaluation ===
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        trained_model.eval()
        all_true_labels = []
        all_predicted_labels = []
        correct_predictions = 0
        total_samples = 0

        with torch.no_grad():
            for images, labels, _ in test_loader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = trained_model(images)
                _, predicted_labels = torch.max(outputs.data, 1)
                total_samples += labels.size(0)
                correct_predictions += (predicted_labels == labels).sum().item()
                all_true_labels.extend(labels.cpu().numpy())
                all_predicted_labels.extend(predicted_labels.cpu().numpy())

        accuracy = 100 * correct_predictions / total_samples
        print(f"\n--- Evaluation Results for {nclasses} classes at {res}x{res} ---")
        print(f"Total Test Samples: {total_samples}")
        print(f"Correct Predictions: {correct_predictions}")
        print(f"Accuracy on the test set: {accuracy:.2f}%")
        print(f"-------------------------------------------------------------")

        # Store in results list
        results.append({
            'num_classes': nclasses,
            'resolution': (res, res),
            'test_samples': total_samples,
            'correct_predictions': correct_predictions,
            'accuracy': accuracy,
            'model_path': model_path,
            'true_labels': all_true_labels,
            'predicted_labels': all_predicted_labels
        })

# Create dataframe
results_df = pd.DataFrame(results)

# Save to CSV (without labels lists) for quick overview
results_df.drop(['true_labels', 'predicted_labels'], axis=1).to_csv('experiment_summary.csv', index=False)

# Save the full object with labels for deeper analysis
results_df.to_pickle('experiment_results.pkl')


In [ ]:
import pandas as pd

df = pd.read_pickle('experiment_results.pkl')

# Example: filter for 50 classes at 64x64
exp = df[(df['num_classes'] == 50) & (df['resolution'] == (64, 64))].iloc[0]

# Reuse your labels for confusion matrix:
ML.plot_confusion_matrix(
    y_true=exp['true_labels'],
    y_pred=exp['predicted_labels'],
    classes=[f'Class {i}' for i in range(exp['num_classes'])],
    normalize=False
)
plt.show()

# Show incorrect predictions using your original code:
model = GlyphClassifier(exp['num_classes'], resolution=exp['resolution'])
model.load_state_dict(torch.load(exp['model_path']))
model.to(device)

ML.show_incorrect_predictions(model, test_loader, num_classes=exp['num_classes'], max_display=10, device=device)

In [ ]:

#  resnet-50, pre-trained on ImageNet
